# 04 - Sensing And State Estimation

## What / Why / How

**What we are trying to do:** Estimate hidden robot state by fusing noisy odometry-like predictions and noisy position measurements.

**Why this matters:** A robot almost never knows the truth directly. State estimation is the bridge between imperfect sensors and reliable control.

**How we will do it:** Generate noisy motion data, run a 1D Kalman filter, and compare raw measurement error against filtered estimate error.

## Goal

Robots rarely know their true state. They estimate it from noisy sensors.

You will implement a 1D Kalman filter that fuses:

- Odometry-like velocity commands
- Noisy position measurements
- A simple motion model

In [ ]:
import math
import random
from collections import defaultdict

import numpy as np

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see the plot: pip install -r requirements.txt")

## Simulate Noisy Motion

In [ ]:
rng = np.random.default_rng(7)
dt = 0.1
steps = 80
true_x = 0.0
true_v = 0.7
process_sigma = 0.03
measurement_sigma = 0.25

true_positions = []
odom_velocities = []
measurements = []
for _ in range(steps):
    true_v = 0.7 + rng.normal(0, 0.02)
    true_x += true_v * dt + rng.normal(0, process_sigma)
    odom_v = true_v + rng.normal(0, 0.08)
    z = true_x + rng.normal(0, measurement_sigma)
    true_positions.append(true_x)
    odom_velocities.append(odom_v)
    measurements.append(z)

true_positions = np.array(true_positions)
odom_velocities = np.array(odom_velocities)
measurements = np.array(measurements)
print("first measurements:", measurements[:5])

## Kalman Filter

In [ ]:
x_hat = 0.0
P = 1.0
Q = process_sigma ** 2
R = measurement_sigma ** 2
estimates = []
uncertainties = []

for u, z in zip(odom_velocities, measurements):
    # Predict
    x_hat = x_hat + u * dt
    P = P + Q

    # Update
    K = P / (P + R)
    x_hat = x_hat + K * (z - x_hat)
    P = (1 - K) * P

    estimates.append(x_hat)
    uncertainties.append(P)

estimates = np.array(estimates)
rmse_measurement = np.sqrt(np.mean((measurements - true_positions) ** 2))
rmse_filter = np.sqrt(np.mean((estimates - true_positions) ** 2))
print("measurement RMSE:", round(float(rmse_measurement), 3))
print("filter RMSE:", round(float(rmse_filter), 3))

In [ ]:
if HAS_PLOT:
    t = np.arange(steps) * dt
    plt.figure(figsize=(9, 3))
    plt.plot(t, true_positions, label="true")
    plt.scatter(t, measurements, s=15, alpha=0.5, label="measurements")
    plt.plot(t, estimates, label="Kalman estimate")
    plt.xlabel("time [s]")
    plt.ylabel("x")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
else:
    plot_unavailable()
    print("last estimate:", estimates[-1], "true:", true_positions[-1])

## What To Notice

The filter is not magic. It is a disciplined compromise between prediction and measurement:

- If measurements are noisy, trust the model more.
- If the model is uncertain, trust measurements more.
- Real robots use higher-dimensional versions: EKF, UKF, factor graphs, visual-inertial odometry, and SLAM backends.

## Exercises

1. Increase `measurement_sigma`. Does the Kalman gain become more cautious?
2. Increase `process_sigma`. Does the filter trust measurements more?
3. Extend the state to include velocity.